# 05.01 — Decision Framework: Kapan Pakai LLM, Kapan SLM

**Tujuan**: setelah jalan modul 02-04, kamu sudah punya intuisi. Sekarang formalisasi jadi **decision framework** yang bisa kamu pakai di proyek nyata.

**Prasyarat**: modul 04 lulus.

**Output**: function `recommend_model()` yang bisa kamu pakai untuk evaluasi cepat sebuah ide proyek.

## 0. Bootstrap (jalankan pertama)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

## 1. 5 pertanyaan yang harus kamu jawab

Sebelum pilih model, jawab 5 pertanyaan ini tentang proyek mu:

1. **Privacy/data sensitivity** — apakah data tidak boleh keluar dari server kamu?
2. **Task complexity** — apakah butuh multi-step reasoning, math, atau coding non-trivial?
3. **Task narrowness** — apakah task spesifik (klasifikasi, extraction) dan ada labeled data?
4. **Volume & latency budget** — berapa request/hari? p95 latency target?
5. **Offline requirement** — harus jalan tanpa internet?

## 2. Function decision framework

In [ ]:
from dataclasses import dataclass

@dataclass
class ProjectSpec:
    name: str
    data_sensitive: bool          # data privacy hard requirement?
    needs_offline: bool           # harus jalan tanpa internet?
    complex_reasoning: bool       # butuh multi-step reasoning / math / code?
    narrow_task: bool             # task spesifik (klasifikasi, NER, extraction)?
    has_labeled_data: bool        # ada training data labeled?
    requests_per_day: int
    latency_p95_ms: int           # latency budget

def recommend(spec: ProjectSpec) -> dict:
    """Decision logic: return {choice, reason}"""
    # 1. Offline = no choice → harus SLM
    if spec.needs_offline:
        return {"choice": "SLM lokal (quantized)", "reason": "offline requirement → no API"}

    # 2. Data sensitive → SLM atau fine-tuned
    if spec.data_sensitive:
        if spec.narrow_task and spec.has_labeled_data:
            return {"choice": "SLM fine-tuned (lokal)", "reason": "privacy + narrow task + ada data → fine-tune wins"}
        return {"choice": "SLM lokal (general, quantized)", "reason": "privacy hard requirement → tidak boleh API"}

    # 3. Complex reasoning → harus LLM besar
    if spec.complex_reasoning:
        return {"choice": "LLM API (Llama-3.3-70b atau yang setara)", "reason": "task butuh reasoning → SLM kecil tidak cukup"}

    # 4. Narrow task + labeled data + volume tinggi → fine-tuned SLM
    if spec.narrow_task and spec.has_labeled_data and spec.requests_per_day > 10_000:
        return {"choice": "SLM fine-tuned (lokal)", "reason": "narrow + data + volume tinggi → fine-tune jauh lebih murah"}

    # 5. Latency ketat → Groq atau lokal
    if spec.latency_p95_ms < 500:
        if spec.requests_per_day < 10_000:
            return {"choice": "LLM API (Groq, low-latency)", "reason": "latency ketat, volume rendah → Groq sweet spot"}
        return {"choice": "SLM lokal (low latency, no network)", "reason": "latency ketat + volume tinggi → self-host"}

    # Default
    return {"choice": "LLM API (default — Groq llama-3.1-8b)", "reason": "start simple, optimize later"}

print("recommend() siap dipakai.")

## 3. Jalankan di 6 skenario nyata

In [ ]:
scenarios = [
    ProjectSpec(
        name="Chatbot customer service e-commerce",
        data_sensitive=False, needs_offline=False,
        complex_reasoning=False, narrow_task=False, has_labeled_data=False,
        requests_per_day=5_000, latency_p95_ms=2_000,
    ),
    ProjectSpec(
        name="Sentiment analysis 50k komentar IG/hari",
        data_sensitive=False, needs_offline=False,
        complex_reasoning=False, narrow_task=True, has_labeled_data=True,
        requests_per_day=50_000, latency_p95_ms=1_000,
    ),
    ProjectSpec(
        name="Asisten medis di klinik (data pasien)",
        data_sensitive=True, needs_offline=False,
        complex_reasoning=True, narrow_task=False, has_labeled_data=False,
        requests_per_day=500, latency_p95_ms=3_000,
    ),
    ProjectSpec(
        name="Coding assistant internal developer",
        data_sensitive=False, needs_offline=False,
        complex_reasoning=True, narrow_task=False, has_labeled_data=False,
        requests_per_day=10_000, latency_p95_ms=2_500,
    ),
    ProjectSpec(
        name="App mobile offline transkripsi audio",
        data_sensitive=True, needs_offline=True,
        complex_reasoning=False, narrow_task=True, has_labeled_data=False,
        requests_per_day=100, latency_p95_ms=10_000,
    ),
    ProjectSpec(
        name="NER (extraction nama orang) dari arsip dokumen",
        data_sensitive=True, needs_offline=False,
        complex_reasoning=False, narrow_task=True, has_labeled_data=True,
        requests_per_day=1_000, latency_p95_ms=5_000,
    ),
]

for s in scenarios:
    rec = recommend(s)
    print(f"📌 {s.name}")
    print(f"   → {rec['choice']}")
    print(f"   karena: {rec['reason']}\n")

## 4. Analisis biaya at scale

Salah satu poin paling penting yang sering dilupakan: biaya **non-linear** dengan volume. Plot biaya/bulan vs request/hari untuk LLM API vs hosting SLM.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from utils.plotting import COLORS, setup_style
setup_style()

# Asumsi: biaya per request Groq ~$0.0002 (avg untuk task kecil), VPS GPU ~$30/bulan
COST_PER_REQUEST_API = 0.0002
VPS_FIXED_MONTHLY = 30  # USD untuk VPS small

x_per_day = np.logspace(2, 7, 50)  # 100 sampai 10M request/hari
api_cost_monthly = x_per_day * 30 * COST_PER_REQUEST_API
vps_cost_monthly = np.full_like(x_per_day, VPS_FIXED_MONTHLY)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(x_per_day, api_cost_monthly, label="LLM API (per-request)", color=COLORS["llm"], lw=2)
ax.loglog(x_per_day, vps_cost_monthly, label="SLM self-hosted (fixed VPS)", color=COLORS["slm_q4"], lw=2)

# Titik break-even
breakeven = VPS_FIXED_MONTHLY / (30 * COST_PER_REQUEST_API)
ax.axvline(breakeven, color="gray", ls="--", alpha=0.5)
ax.text(breakeven, 1, f"break-even\n~{breakeven:.0f}/hari", ha="center", fontsize=9)

ax.set_xlabel("Request per hari")
ax.set_ylabel("Biaya USD per bulan")
ax.set_title("Biaya bulanan: LLM API vs SLM self-hosted (asumsi ~$0.0002/req, VPS $30/mo)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nBreak-even di ~{breakeven:.0f} request/hari.")
print("Di bawah itu → API lebih murah. Di atas itu → self-hosted lebih murah.")
print("\nTapi ingat: biaya engineering maintain self-hosted TIDAK termasuk di angka ini.")

## 5. Hybrid pattern: routing & cascade

Banyak production system pakai **kombinasi** untuk optimasi cost/quality:

### Pattern 1: Routing (SLM klasifier intent → routing ke LLM tepat)
```
user query
  → DistilBERT klasifikasi intent (cheap, fast)
    → intent=simple_FAQ   → SLM lokal jawab
    → intent=needs_reasoning → Groq llama-8b
    → intent=complex_legal → Groq llama-70b (mahal tapi worth it)
```

### Pattern 2: Cascade (SLM coba dulu, kalau confidence rendah → LLM)
```
user query
  → DistilBERT jawab + confidence score
    → confidence > 0.85: return jawaban SLM (cheap)
    → confidence ≤ 0.85: eskalasi ke LLM
```

### Pattern 3: Distill (LLM jadi teacher untuk SLM)
```
Offline: pakai Groq llama-70b generate 50k contoh teks→label berkualitas tinggi.
Online: fine-tune DistilBERT pakai 50k contoh itu. Production cuma DistilBERT.
```

Ketiga pattern ini cara mainstream production system kombinasi-kan LLM + SLM.

## 6. Kesalahan klasik yang harus dihindari

1. **"Kita pakai LLM untuk sentiment analysis"** — sering 100x overkill. DistilBERT fine-tuned biasanya menang accuracy + cost + latency.
2. **"Kita self-host Llama-70B"** — kecuali tim mu MLOps-ready dengan GPU cluster, biaya engineering jauh lebih besar dari setahun pakai API.
3. **"SLM lokal pasti privat"** — kalau log model output bocor di /tmp, dependencies ada vuln, atau outbound network ke telemetry: privacy bocor juga.
4. **"Quantization gratis"** — Q4 di task math/code sering drop quality signifikan. A/B test wajib.
5. **Optimasi prematur** — start dengan LLM API (Groq), monitor cost+latency. Migrate ke SLM ketika ada signal jelas (volume/cost/privacy).
6. **Lupa hitung biaya engineer** — "hemat $50/bulan API" yang costing 2 hari engineer/bulan untuk maintain = boncos.

## Refleksi

Tidak ada satu jawaban benar untuk "LLM atau SLM?". Jawabannya **"tergantung"** — dan kamu sekarang tahu **tergantung apa**:

- 5 pertanyaan utama (privacy, complexity, narrowness, volume, offline)
- 3 pattern hybrid kalau jawaban di tengah
- 6 kesalahan klasik untuk dihindari

Kamu sudah punya cukup framework untuk **mempresentasikan pilihan model di rapat tim** dengan justifikasi konkret, bukan cuma feeling.

## Cross-link

Cheatsheet ringkas (printable): [`docs/kapan-pakai-llm-vs-slm.md`](../docs/kapan-pakai-llm-vs-slm.md).

## Lanjut

Closing modul: kerjakan mini-project sendiri pakai framework ini → [02_proyek_mini_pilihanmu.ipynb](02_proyek_mini_pilihanmu.ipynb)